In [ ]:
import os
import math
import random
import pickle
import zipfile
import textwrap
import numpy as np
import seaborn as sns
import plotly.offline as py
import plotly.colors as plc
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

SEED = 69

## Shared Utilities / Helpers

#### Linear algebra helpers:

In [ ]:
def frobenius_norm_sq(A):
    """
    Frobenius norm squared of a matrix.
    """
    A = np.asarray(A, dtype=float)
    return float(np.sum(np.square(A)))

In [ ]:
def l2_norm(v):
    """
    L2 norm of a vector.
    """
    v = np.asarray(v, dtype=float)
    return float(np.linalg.norm(v))

In [ ]:
def orthogonalize(v, basis_vectors):
    """
    Gram-Schmidt orthogonalization + normalization of a vector with respect to a list of basis vectors.

    Parameters:
    v (list of floats): Vector to orthogonalize.
    basis_vectors (list of lists of floats): List of basis vectors.

    Returns:
    numpy.ndarray | None: Orthogonalized unit vector, or None if degenerate.
    """
    v_new = np.asarray(v, dtype=float).copy()

    for b in basis_vectors:
        b_np = np.asarray(b, dtype=float)
        norm_sq = float(np.dot(b_np, b_np))
        if norm_sq < 1e-12:
            continue

        proj = float(np.dot(v_new, b_np)) / norm_sq
        v_new -= proj * b_np

    norm = float(l2_norm(v_new))
    if norm < 1e-12:
        return None

    return v_new / norm


In [ ]:
def power_iteration(matrix, max_iterations=100, tolerance=1e-7, basis_vectors=None):
    """
    Computes largest eigenvalue and corresponding eigenvector using power iteration.

    Parameters:
    matrix (list of lists of floats): Input square matrix.
    max_iterations (int): Number of allowed iterations.
    tolerance (float): Convergence tolerance.
    basis_vectors (list): Optional basis vectors for orthogonalized starts.

    Returns:
    top_eigval (float): Largest eigenvalue.
    top_eigvec (numpy.ndarray): Corresponding eigenvector.
    """
    A = np.asarray(matrix, dtype=float)
    n = A.shape[1]

    if basis_vectors is None:
        basis_vectors = []

    # initialize random vector and orthogonalize (this will be prev eigenvector)
    v_prev = np.random.standard_normal(n)
    v_prev = orthogonalize(v_prev, basis_vectors)

    if v_prev is None:
        fallback = np.random.standard_normal(n)
        fallback /= max(float(l2_norm(fallback)), 1e-12)
        return 0.0, fallback

    for _ in range(max_iterations):
        # matrix-vector multiplication (this is the current eigenvector)
        v_curr = A @ v_prev

        # keep iterating orthogonal to known basis when provided
        if basis_vectors:
            v_curr = orthogonalize(v_curr, basis_vectors)
            if v_curr is None:
                return 0.0, v_prev

        # normalize curr eigenvector and handle zero vec case
        norm = float(l2_norm(v_curr))
        if norm < 1e-12:
            return 0.0, v_prev
        v_curr = v_curr / norm

        # sign-invariant convergence check
        delta = min(
            float(l2_norm(v_curr - v_prev)),
            float(l2_norm(v_curr + v_prev)),
        )

        # update previous eigenvector
        v_prev = v_curr

        # check convergence (delta should approach 0)
        if delta < tolerance:
            break

    # compute corresponding eigenvalue (Rayleigh quotient)
    top_eigval = float(v_prev @ (A @ v_prev))
    top_eigvec = v_prev

    return top_eigval, top_eigvec

In [ ]:
def top_k_eigenpairs(
    matrix,
    max_k=30,
    max_subspace_iterations=500,
    tolerance=1e-9,
):
    """
    Top k eigenpairs of symmetric A via simultaneous (subspace) iteration + Rayleigh–Ritz.

    Orthonormal eigenvectors; avoids deflation drift when k is large.

    Parameters:
    matrix: Square symmetric matrix (list of lists or ndarray).
    max_k: Number of eigenpairs to return (capped at matrix size).
    max_subspace_iterations: QR iterations on span(A^k Q).
    tolerance: Stop when max off-diagonal of the Rayleigh matrix H = QᵀAQ is below this.

    Returns:
    eigvals (np.ndarray): Top eigenvalues in descending order, shape (p,).
    eigvecs (np.ndarray): Corresponding eigenvectors as columns, shape (n, p).
    """
    A = np.asarray(matrix, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("matrix must be square")

    n = A.shape[0]
    p = min(max_k, n)
    if p == 0:
        return np.empty((0,), dtype=float), np.empty((n, 0), dtype=float)

    # initialize random orthonormal subspace basis
    Q, _ = np.linalg.qr(np.random.standard_normal((n, p)), mode="reduced")

    # iterate QR iterations on span(A^k Q)
    # until reduced problem is nearly diagonal
    for _ in range(max_subspace_iterations):

        Z = A @ Q
        Q_new, _ = np.linalg.qr(Z, mode="reduced")

        # Rayleigh–Ritz projection
        H = Q_new.T @ A @ Q_new
        H = 0.5 * (H + H.T)

        # check convergence
        off = float(np.max(np.abs(H - np.diag(np.diag(H)))))
        Q = Q_new
        if off < tolerance:
            break

    # final Rayleigh–Ritz projection
    H = Q.T @ A @ Q
    H = 0.5 * (H + H.T)

    # compute eigenvalues and eigenvectors
    eigvals_h, eigvecs_h = np.linalg.eigh(H)

    # sort eigenvalues in descending order
    idx = np.argsort(eigvals_h)[::-1]
    eigvals = eigvals_h[idx][:p]

    # reduce eigenvectors to top-k
    vecs_reduced = eigvecs_h[:, idx][:, :p]
    eigvecs = Q @ vecs_reduced

    # normalize eigenvectors
    col_norms = np.linalg.norm(eigvecs, axis=0, keepdims=True)
    col_norms[col_norms < 1e-12] = 1.0
    eigvecs = eigvecs / col_norms

    return eigvals, eigvecs

In [ ]:
def sing_val_decomp(matrix, r=None, mode="full"):
    """
    Manually performs singular value decomposition (SVD) on a given matrix.
    
    Parameters:
    matrix (list of lists of floats): Input matrix to decompose.
    r (int): Number of singular values to compute.
    mode (str): "full" or "partial".

    Returns:
    U   (list of lists of floats): Left singular vectors.
    S   (list of lists of floats): Singular values.
    V_T (list of lists of floats): Right singular vectors (transposed).
    """
    M = np.asarray(matrix, dtype=float)
    m, n = M.shape
    r = min(m, n) if r is None else min(int(r), m, n)

    # compute right-side positive semidefinite matrix
    PSD_R = M.T @ M

    eigenvalues, eigenvectors = top_k_eigenpairs(PSD_R, max_k=r)

    # compute U (m, r), V (n, r), and S (r, r)
    if mode == "partial":
        U, V_T = None, None
        S = np.zeros((r, r), dtype=float)
    elif mode == "full":
        U = np.zeros((m, r), dtype=float)
        V = np.zeros((n, r), dtype=float)
        S = np.zeros((r, r), dtype=float)
    else:
        raise ValueError("Invalid mode. Must be 'full' or 'partial'.")

    eps = 1e-10
    for i in range(r):
        # compute sigma_i
        sigma = max(float(eigenvalues[i]), 0.0) ** 0.5
        S[i, i] = sigma if sigma > eps else 0.0

        if mode == "full":
            # compute V_i
            eigvec_i = np.asarray(eigenvectors[:, i], dtype=float)
            V[:, i] = eigvec_i

            # compute U_i = (M * V_i) / sigma_i
            if sigma > eps:
                U[:, i] = (M @ eigvec_i) / sigma

    if mode == "full":
        V_T = V.T

    return U, S, V_T

#### Evaluation helpers:

In [ ]:
def wrap_text(text, width=16):
    """Wraps text to a given width."""
    return ["\n".join(textwrap.wrap(s, width=width)) for s in text]

In [ ]:
def interpolate_sequence(data_sequence, num_interpolations=10):
    """
    Interpolates a sequence of matrices along a time axis using cubic splines.
    
    Parameters:
    data_sequence (list of numpy.ndarray): List of matrices representing data at different time steps.
    num_interpolations (int): Number of interpolation steps between each pair of matrices.
    
    Returns:
    interpolated_data (numpy.ndarray): Interpolated data sequence of shape (total_steps, N, D).
    """
    # convert list of decade matrices to 3D array
    data = np.asarray(data_sequence, dtype=float)
    T, N, D = data.shape

    # parameterize time steps t from 0 to T-1
    t = np.arange(T)

    # compute total number of interpolation steps
    # and create dense time grid for interpolation
    total_steps = (T - 1) * (num_interpolations + 1) + 1
    t_interp = np.linspace(0, T-1, total_steps)

    # initialize cubic spline interpolation on time axis
    # and set natural boundary conditions (so curves don't overshoot)
    spline = CubicSpline(t, data, axis=0, bc_type="natural")
    return spline(t_interp)

In [ ]:
def cosine_similarity(v1, v2):
    """
    Computes the cosine similarity between two vectors.
    
    Parameters:
    v1, v2: Input vectors.

    Returns:
    float: Cosine similarity between the two vectors.
    """
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)

    # calculate the dot product
    dot = float(np.dot(v1, v2))
    
    # calculate L2 norm for each vector
    norm_v1 = float(np.linalg.norm(v1))
    norm_v2 = float(np.linalg.norm(v2))

    # handle division by zero if any of the vectors have zero magnitude
    if norm_v1 == 0 or norm_v2 == 0:
        return 0

    # calculate cosine similarity
    return dot / (norm_v1 * norm_v2)

In [ ]:
def jaccard_similarity(set1, set2):
    """
    Calculate the Jaccard similarity between two sets.
    
    Parameters:
    set1, set2: Sets representing co-occurring words for a target word.

    Returns:
    float: Jaccard similarity between the two sets.
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

## Step 1: Preprocessing

#### Load HistWords dataset:

In [ ]:
# # dataset obtained from here: https://github.com/williamleif/histwords

# # load dataset here:
# !git clone https://github.com/williamleif/histwords.git
# !wget -O all_english_embeddings.zip "http://snap.stanford.edu/historical_embeddings/eng-all_sgns.zip"

# # unzip loaded dataset
# with zipfile.ZipFile("all_english_embeddings.zip", 'r') as zip_ref:
#     zip_ref.extractall("all_english_embeddings")

In [ ]:
# base dir for embeddings
base_dir = "all_english_embeddings/sgns"
print(os.getcwd()) # debugging

In [ ]:
# for loading historical embeddings from .npy files
def load_embeddings(file_path):
    return np.load(file_path)

#### Obtain sample of words from dataset:

In [ ]:
# specify decades of interest
decades = ['1880', '1890', '1900', '1910', '1920', '1930', '1940', '1950', '1960', '1970', '1980']
vocab_dict = {}
embeddings_dict = {}
random.seed(SEED)       # set desired seed here
sample_size = 500       # set desired sample size here

In [ ]:
# load and optionally sample each decade's embeddings and vocabulary
for decade in decades:
    # load embeddings
    embedding_path = os.path.join(base_dir, f"{decade}-w.npy")
    embeddings = load_embeddings(embedding_path)
    
    # load vocabulary
    vocab_path = os.path.join(base_dir, f"{decade}-vocab.pkl")
    with open(vocab_path, 'rb') as f:
        vocab = pickle.load(f)
    
    # ensure embeddings and vocab are same size
    assert len(embeddings) == len(vocab), f"Mismatch in size for {decade}"

    # filter out any numerical values incorrectly counted as "words"
    filtered_vocab = []
    filtered_embeddings = []
    for word, embedding in zip(vocab, embeddings):
        if not any(char.isdigit() for char in word):
            filtered_vocab.append(word)
            filtered_embeddings.append(embedding)

    # fill dicts
    vocab_dict[decade] = filtered_vocab
    embeddings_dict[decade] = np.array(filtered_embeddings)

#### Filter subsamples by intersection and non-zero vectors across decades:

In [ ]:
# obtain common vocabulary across all decades
candidate_vocabs = set(vocab_dict[decades[0]])
for decade in decades[1:]:
    candidate_vocabs.intersection_update(vocab_dict[decade])

# convert to list for indexing, sort for determinism, then shuffle for reproducible randomness
candidate_vocabs = list(candidate_vocabs)
candidate_vocabs.sort()
random.shuffle(candidate_vocabs)

In [ ]:
# precompute word to index dictionaries for each decade
word_to_idx_dict = {}
for decade in decades:
    vocab = vocab_dict[decade]
    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    word_to_idx_dict[decade] = word_to_idx

valid_words = []
zero_vector = np.zeros(embeddings_dict[decades[0]].shape[1])

# filter out words with zero vectors across all decades
for word in candidate_vocabs:
    has_zero_vector = False
    for decade in decades:
        idx = word_to_idx_dict[decade][word]
        embedding = embeddings_dict[decade][idx]
        if np.array_equal(embedding, zero_vector):
            has_zero_vector = True
            break

    # add word to valid_words if it has non-zero vectors across all decades
    if not has_zero_vector:
        valid_words.append(word)

    # stop if we have enough valid words
    if len(valid_words) == sample_size:
        break

print(f"Number of valid words (non-zero embeddings in all decades): {len(valid_words)}")

# update vocab_dict and embeddings_dict
for decade in decades:
    embeddings = embeddings_dict[decade]
    word_to_idx = word_to_idx_dict[decade]
    
    # obtain embeddings for valid_words
    indices = [word_to_idx[word] for word in valid_words]
    filtered_embeddings = embeddings[indices]
    
    # update dicts with filtered values
    vocab_dict[decade] = valid_words
    embeddings_dict[decade] = filtered_embeddings

# rebuild word to index dict after subsampling
word_to_idx_dict = {
    decade: {w: i for i, w in enumerate(vocab_dict[decade])}
    for decade in decades
}

In [ ]:
# debugging
for decade, embeddings in embeddings_dict.items():
    print(f"{decade}'s embeddings:")
    print(f"shape: {embeddings.shape}") # display shape of embeddings per decade
    print(embeddings_dict[decade][:5])  # display first n rows of embeddings per decade
    print()

## Step 2: Standardization

In [ ]:
def standardize(data):
    """
    Standardizes vector data by centering (subtracting mean) and scaling (dividing by standard deviation).

    Parameters:
    data (numpy.ndarray): Input data matrix of shape (num_vectors, num_dimensions).

    Returns:
    standardized_data (numpy.ndarray): Standardized data with zero mean and unit variance.
    mean_array (numpy.ndarray): Computed mean matrix.
    std_array (numpy.ndarray): Computed standard deviation matrix.
    """
    data = np.asarray(data, dtype=float)

    # compute mean and std for each dimension
    # using ddof=1 for sample std, Bessel's correction (unbiased estimator)
    mean_array = np.mean(data, axis=0)
    std_array = np.std(data, axis=0, ddof=1)

    # center and scale data
    standardized_data = (data - mean_array) / np.where(std_array != 0, std_array, 1.0)
    return standardized_data, mean_array, std_array

#### Center each decade's high-dim point cloud to zero mean and scale to unit variance:

In [ ]:
std_embeddings_dict = {}
for decade, embeddings in embeddings_dict.items():
    standardized_embeddings = standardize(embeddings_dict[decade])[0]
    std_embeddings_dict[decade] = standardized_embeddings

In [ ]:
# debugging: per decade — non-centered vs centered mean (first 5 dimensions)
for decade in decades:
    mean_raw = standardize(embeddings_dict[decade])[1]
    mean_std = standardize(std_embeddings_dict[decade])[1]
    print(f"Embedding means before centering ({decade}'s):", mean_raw[:5])
    print(f"Embedding means after centering ({decade}'s):", mean_std[:5])
    print()

## Step 3: Alignment

In [ ]:
def procrustes_residual(A, B, norm_A, norm_B):
    """
    Manually computes the Procrustes residual between two matrices using singular values.
    
    Parameters:
    A (np.ndarray): First matrix, shape (n, d).
    B (np.ndarray): Second matrix, shape (n, d).
    norm_A (float): Frobenius norm squared of A.
    norm_B (float): Frobenius norm squared of B.
    
    Returns:
    float: Procrustes residual between A and B.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    M = B.T @ A

    # sum of singular values of cross-covariance (same as trace term in optimal orthogonal alignment)
    trace_sigma = float(np.sum(np.linalg.svd(M, full_matrices=False, compute_uv=False)))

    # compute residual
    residual = norm_A + norm_B - 2.0 * trace_sigma
    return max(0.0, float(residual))

In [ ]:
def build_pairwise_residual_matrix(embeddings_dict, decades, norm_cache):
    """
    Builds a pairwise residual matrix for a given set of decades.
    
    Parameters:
    embeddings_dict (dict): Dictionary of embeddings for each decade.
    decades (list of str): List of decades to compute pairwise residuals for.
    norm_cache (dict): Dictionary of Frobenius norms for each decade.
    r (int): Number of singular values used for partial SVD approximation.

    Returns:
    R (numpy.ndarray): Pairwise residual matrix.
    """
    m = len(decades)
    R = np.zeros((m, m), dtype=float)

    # cache per-decade matrices/transposes once to reduce repeated conversion and transpose overhead
    mats = {d: np.asarray(embeddings_dict[d], dtype=float) for d in decades}
    mats_T = {d: mats[d].T for d in decades}

    for i in range(m):
        d1 = decades[i]
        A = mats[d1]
        for j in range(i + 1, m):
            d2 = decades[j]
            B = mats[d2]
            B_T = mats_T[d2]

            residual = procrustes_residual(A, B, norm_cache[d1], norm_cache[d2])

            R[i, j] = residual
            R[j, i] = residual
    return R

In [ ]:
def compute_transform_matrix(A, B):
    """
    Orthogonal Procrustes rotation W minimizing:
    ||A W - B||_F (same width).

    After manual SVD, projects W = V Uᵀ onto the nearest orthogonal matrix so
    ‖A W‖_F = ‖A‖_F when W is orthogonal.

    Parameters:
    A (np.ndarray): Matrix to transform.
    B (np.ndarray): Target matrix.

    Returns:
    W (np.ndarray): Orthogonal Procrustes rotation matrix.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    M = A.T @ B

    U, _, V_T = np.linalg.svd(M, full_matrices=False, compute_uv=True)
    W = U @ V_T
    return W

In [ ]:
# compute Frobenius norms for each decade
norm_cache = {
    decade: frobenius_norm_sq(embeddings)
    for decade, embeddings in std_embeddings_dict.items()
}

In [ ]:
# compute pairwise Procrustes residuals
R = build_pairwise_residual_matrix(std_embeddings_dict, decades, norm_cache)

# compute anchor scores
anchor_scores = {} # decade to score dict
for i, decade in enumerate(decades):
    anchor_scores[decade] = sum(R[i])

# pick lowest scoring residual decade as the anchor
anchor_decade = min(anchor_scores, key=anchor_scores.get)

In [ ]:
# compute alignment transforms
transforms = {}
B = std_embeddings_dict[anchor_decade]
for decade, embeddings in std_embeddings_dict.items():
    if decade == anchor_decade:
        transforms[decade] = None
        continue
    A = embeddings
    W = compute_transform_matrix(A, B)
    transforms[decade] = W

In [ ]:
# apply alignment transforms to align all embeddings to anchor decade
aligned_embeddings_dict = {}
for decade, embeddings in std_embeddings_dict.items():
    A = embeddings
    W = transforms[decade]
    if W is None:
        aligned_embeddings_dict[decade] = A.copy() # deep copy of matrix to avoid modifying original
    else:
        aligned_embeddings_dict[decade] = A @ W

In [ ]:
anchor_matrix = np.asarray(std_embeddings_dict[anchor_decade], dtype=float)
print(f"Anchor decade: {anchor_decade}\n")

# debugging: per decade — alignment error vs anchor before and after Procrustes
pre_alignment_errors = []
post_alignment_errors = []
for decade in decades:
    A_pre = np.asarray(std_embeddings_dict[decade], dtype=float)
    A_post = np.asarray(aligned_embeddings_dict[decade], dtype=float)
    pre_err = frobenius_norm_sq(A_pre - anchor_matrix)
    post_err = frobenius_norm_sq(A_post - anchor_matrix)
    pre_alignment_errors.append(pre_err)
    post_alignment_errors.append(post_err)
    print(f"Alignment error before ({decade} vs {anchor_decade}): {pre_err:.6f}")
    print(f"Alignment error after  ({decade} vs {anchor_decade}): {post_err:.6f}")
    print()

improved_count = sum(
    1 for pre, post in zip(pre_alignment_errors, post_alignment_errors) if post < pre
)
avg_pre = sum(pre_alignment_errors) / len(pre_alignment_errors)
avg_post = sum(post_alignment_errors) / len(post_alignment_errors)

print(f"Decades improved (lower error after): {improved_count}/{len(decades)}")
print(f"Average pre-alignment squared error:  {avg_pre:.6f}")
print(f"Average post-alignment squared error: {avg_post:.6f}")

## Step 4: PCA

#### Calculate covariance matrix and find top-k eigenvalues:

In [ ]:
def compute_cv_matrix(data):
    """
    Manually computes covariance matrix of input data.
    
    Parameters:
    data (list of lists of floats): Centered data matrix of shape (num_vectors, num_dimensions).
    
    Returns:
    covariance_matrix (list of lists of floats): Covariance matrix of shape (num_dimensions, num_dimensions).
    """
    A = np.asarray(data, dtype=float)
    num_vectors = A.shape[0]
    covariance_matrix = (A.T @ A) / (num_vectors - 1)
    return covariance_matrix

In [ ]:
# combine embeddings from all decades into one dataset for unified coordinate space
combined_embeddings = np.vstack([aligned_embeddings_dict[decade] for decade in decades])

# sanity check: mean of each decade's embeddings should be close to zero
mu = combined_embeddings.mean(axis=0)
is_centered = np.allclose(mu, 0.0, atol=1e-8) # we can relax atol if needed
assert is_centered, "Combined embeddings are not centered"
print(f"Is centered? {is_centered}")

In [ ]:
# find cv matrix of combined embeddings
cv_matrix = compute_cv_matrix(combined_embeddings)

# sanity check: the cv matrix should be symmetric
is_symmetric = np.allclose(cv_matrix, cv_matrix.T)
assert is_symmetric, "CV matrix is not symmetric"
print(f"Is symmetric? {is_symmetric}")

In [ ]:
# debugging
largest_eigenvalue, eigenvector = power_iteration(cv_matrix)
print("Largest eigenvalue:", largest_eigenvalue)

# debugging
k = 3
eigenvalues, eigenvectors = top_k_eigenpairs(cv_matrix, k)
print(f"Largest {k} eigenvalues:", eigenvalues)

## Step 5: Projection

In [ ]:
# project each decade's aligned embeddings onto the common pooled PC basis (columns of eigenvectors)
proj_embeddings_dict = {} # this dict should hold embeddings of words in R3
for decade in decades:
    projected_embeddings = aligned_embeddings_dict[decade] @ eigenvectors
    proj_embeddings_dict[decade] = projected_embeddings

In [ ]:
# check for matching vocab and projected embedding dict lengths (debugging)
for decade in decades:
    assert len(vocab_dict[decade]) == len(embeddings_dict[decade]), f"Mismatch in vocab and embeddings for {decade}"

In [ ]:
print("Keys in projected embeddings dict:", proj_embeddings_dict.keys())

## Step 6: Visualization

In [ ]:
# initial plot setup and parameters

text_size = 8
marker_size = 3
num_interps = 20  # adjust for smoother transitions here

# fixed axis ranges
axes_bounds = 10
x_min, x_max = -axes_bounds, axes_bounds
y_min, y_max = -axes_bounds, axes_bounds
z_min, z_max = -axes_bounds, axes_bounds

axes_intervals = [x_min, -axes_bounds/2, 0, axes_bounds/2, x_max]

# camera zoom control
camera = dict(
    eye=dict(x=1.5, y=1.5, z=1.5)  # adjust for zoom level (default is x=1, y=1, z=1)
)

# stack all projected embeddings into single array
# then interpolate point clouds along time axis
proj_embeddings_array = [proj_embeddings_dict[d] for d in decades]
smoothed_data = interpolate_sequence(proj_embeddings_array, num_interpolations=num_interps)

#### Timeline plot:

In [ ]:
n_timeline_frames = smoothed_data.shape[0]
nd_decades = len(decades)
frames = []

# create frames for each interpolated point cloud
for frame_idx, frame_data in enumerate(smoothed_data):
    # map frame index to nearest decade label (uniform t grid from interpolate_sequence)
    if n_timeline_frames <= 1:
        decade_idx = 0
    else:
        decade_idx = int(round(frame_idx * (nd_decades - 1) / (n_timeline_frames - 1)))

    # clamp decade index to valid range
    decade_idx = max(0, min(decade_idx, nd_decades - 1))
    curr_decade = decades[decade_idx]

    frame = go.Frame(
        data=[
            go.Scatter3d(
                x=frame_data[:, 0],
                y=frame_data[:, 1],
                z=frame_data[:, 2],
                mode='markers+text',
                marker=dict(size=marker_size, color='blue'),
                text=vocab_dict[curr_decade],
                textfont=dict(size=text_size),
                name=f"Frame {frame_idx}"
            )
        ],
        name=f"Frame {frame_idx}"
    )
    frames.append(frame)

# build slider steps to control tick density
_slider_steps = []
for k in range(n_timeline_frames):
    if n_timeline_frames <= 1:
        _di = 0
    else:
        _di = int(round(k * (nd_decades - 1) / (n_timeline_frames - 1)))
    _di = max(0, min(_di, nd_decades - 1))

    _slider_steps.append({
        "args": [[f"Frame {k}"], {
            "frame": {"duration": 100, "redraw": True},
            "mode": "immediate",
            "transition": {"duration": 100}
        }],
        "label": decades[_di],
        "method": "animate",
    })

# define layout for timeline plot
layout_1 = go.Layout(
    title="Per-Decade Positions of Subsampled Words Over 1880-1980 Time Span",
    title_x=0.5,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis=dict(title="PC1", range=[x_min, x_max], tickvals=axes_intervals, autorange=False),
        yaxis=dict(title="PC2", range=[y_min, y_max], tickvals=axes_intervals, autorange=False),
        zaxis=dict(title="PC3", range=[z_min, z_max], tickvals=axes_intervals, autorange=False),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1),
        camera=camera,
    ),
    height=800,
    width=1000,
    sliders=[{
        "steps": _slider_steps,
        "active": 0,
        "currentvalue": dict(visible=False),
        "x": 0.2,
        "xanchor": "left",
        "y": 0,
        "yanchor": "top",
        "len": 0.5
    }],
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {
                    "frame": {"duration": 100, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }],
                "label": "Play",
                "method": "animate"
            },
            {
                "args": [[None], {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0}
                }],
                "label": "Pause",
                "method": "animate"
            },
            {
                "args": [[f"Frame 0"], {  # reset to first interpolated frame
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0}
                }],
                "label": "Reset",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }]
)

# plot word data points
fig_1 = go.Figure(data=[frames[0].data[0]], layout=layout_1, frames=frames)
py.iplot(fig_1)

#### Trajectory plot:

In [ ]:
trajectory_lines = [] # data for trajectory lines of interpolated points
N = 20 # sample size

# create line for each word (representing its interpolated trajectory across decades)
for word_idx in random.sample(range(len(valid_words)), N):
    # get interpolated path of current word across decades
    path = smoothed_data[:, word_idx, :]

    # collect positions of word across decades
    x_spline, y_spline, z_spline = path[:, 0], path[:, 1], path[:, 2]

    # add labels to end of each trajectory line
    text_labels = ["" for _ in range(len(x_spline))] # init empty labels
    text_labels[-1] = valid_words[word_idx] # add label (only at last point of each trajectory)

    # randomize line color for each word's trajectory
    color = "rgb({}, {}, {})".format(random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

    # create lines connecting smoothed points
    trajectory_line = go.Scatter3d(
        x=x_spline,
        y=y_spline,
        z=z_spline,
        mode='lines+text',
        line=dict(color=color, width=2),
        text=text_labels,
        textposition='top center',  # Position of the label
        textfont=dict(size=text_size),
        name=f"{valid_words[word_idx]}"
    )
    trajectory_lines.append(trajectory_line)

# define layout for trajectory plot
layout_2 = go.Layout(
    title="Trajectories of Select Words Over 1880-1980 Time Span",
    title_x=0.5,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis=dict(title="PC1", range=[x_min, x_max], tickvals=axes_intervals, autorange=False),
        yaxis=dict(title="PC2", range=[y_min, y_max], tickvals=axes_intervals, autorange=False),
        zaxis=dict(title="PC3", range=[z_min, z_max], tickvals=axes_intervals, autorange=False),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1),
        camera=camera
    ),
    height=800,
    width=1000,
)

# plot word data trajectories
fig_2 = go.Figure(data=trajectory_lines, layout=layout_2)
py.iplot(fig_2)

## Step 7: Evaluation

#### Quantify trajectorial behavior of select words using Euclidean and Cosine distances:

In [ ]:
# assemble word trajectories across aligned embedding spaces

word_trajectories = {}
for word in valid_words:
    trajectory = []
    for decade in decades:
        idx = word_to_idx_dict[decade][word]
        vec = aligned_embeddings_dict[decade][idx]
        trajectory.append(vec)
    word_trajectories[word] = np.array(trajectory)

print(f"Assembled trajectories for {len(word_trajectories)} words.")

In [ ]:
# calculate metrics

# word to score pairs
path_instability_scores = {} # path instability score for each word (std dev of step magnitudes)
cumulative_dist_scores = {} # cumulative straight-line path length score for each word (Euclidean distance)
displacement_scores = {} # end-to-end angular displacement score for each word (cosine distance)

# calculate shifts for all words across decades
for word, trajectory in word_trajectories.items():
    # find stepwise differences between consecutive decades
    # and stepwise magnitudes
    step_deltas = np.diff(trajectory, axis=0)
    step_magnitudes = np.linalg.norm(step_deltas, axis=1)

    # calculate end-to-end angular displacement
    start_vec = trajectory[0]
    end_vec = trajectory[-1]
    displacement = 1 - cosine_similarity(start_vec, end_vec)

    path_instability_scores[word] = float(np.std(step_magnitudes, ddof=1))
    cumulative_dist_scores[word] = float(np.sum(step_magnitudes))
    displacement_scores[word] = float(displacement)

# sort words by volatility in descending order
sorted_path_instability = sorted(path_instability_scores.items(), key=lambda x: x[1], reverse=True)
sorted_cumulative_dist = sorted(cumulative_dist_scores.items(), key=lambda x: x[1], reverse=True)
sorted_displacement = sorted(displacement_scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
N = 15

# debugging: output top N most path unstable words
for word, score in sorted_path_instability[:N]:
    print(f"Path instability score for '{word}': {score:.4f}")

print()

# debugging: output top N most cumulative L2 displacement words
for word, score in sorted_cumulative_dist[:N]:
    print(f"Cumulative L2 displacement for '{word}': {score:.4f}")

print()

# debugging: output top N most end-to-end displacement words
for word, score in sorted_displacement[:N]:
    print(f"End-to-end displacement for '{word}': {score:.4f}")

#### Quantify changes in neighborhood overlap of select words using k-NN and Jaccard similarity:

In [ ]:
K = 10 # number of nearest neighbors to consider
neighbor_sets = {d: {} for d in decades} # word to per-decade neighbor set pairs
neighbor_distances = {d: {} for d in decades} # word to per-decade neighbor distance pairs

# compute neighborhood sets and cosine distances for each word in each decade
# working in original R300 space

for decade in decades:
    # get aligned embeddings and vocabs for current decade
    X = aligned_embeddings_dict[decade]
    words = vocab_dict[decade]

    # normalize embeddings to unit length
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    Xn = X / norms

    # compute pairwise cosine similarity matrix
    S = Xn @ Xn.T

    # find indices of K nearest neighbors for each word
    knn_idx = np.argsort(S, axis=1)[:, -(K+1):-1]

    # compute neighbor sets and distances for each word
    for i, word in enumerate(words):
        # get indices of K nearest neighbors for current word
        neighbors = {words[j] for j in knn_idx[i]}

        # compute mean cosine distance of current word to its neighbors
        # (distance = 1 - cos similarity)
        avg_dist = 1.0 - np.mean(S[i, knn_idx[i]])

        # store neighbor sets and cosine distances for current word
        neighbor_sets[decade][word] = neighbors
        neighbor_distances[decade][word] = avg_dist

In [ ]:
# compute Jaccard similarity for each word and its neighbors across consecutive decade pairs

jaccard_scores = {} # word to "per-decade Jaccard similarity scores" pairs
dispersion_scores = {} # word to "per-decade neighbor dispersion scores" pairs

for word in valid_words:
    jaccard_scores[word] = []
    dispersion_scores[word] = []
    for i in range(len(decades) - 1):
        # get decade pairs and corresponding neighbor sets and distances
        dec1, dec2 = decades[i], decades[i + 1]
        set1 = neighbor_sets[dec1].get(word, set())
        set2 = neighbor_sets[dec2].get(word, set())
        dist1 = neighbor_distances[dec1].get(word, 0)
        dist2 = neighbor_distances[dec2].get(word, 0)

        # compute Jaccard similarity and dispersion scores for current word
        jaccard_scores[word].append(jaccard_similarity(set1, set2))
        dispersion_scores[word].append((dist1 + dist2) / 2.0)

In [ ]:
N = 20 # num words to display for Jaccard similarity heatmap
selected_words = list(jaccard_scores.keys())[:N]

# create heatmap data
heatmap_data = [jaccard_scores[word] for word in selected_words]
decades_labels = [f"{decades[i]}-{decades[i + 1]}" for i in range(len(decades) - 1)]

# plot heatmap
plt.figure(figsize=(12, 8))

ax = sns.heatmap(heatmap_data, annot=True, cmap="YlGnBu", xticklabels=decades_labels, yticklabels=selected_words)

# font size of axis tick labels
ax.tick_params(axis='x', labelsize=8)
ax.tick_params(axis='y', labelsize=8)

plt.title("Jaccard Similarity Heatmap for Select Words Over 1880-1980 Time Span")
plt.xlabel("Decade Pairs")
plt.ylabel("Words")
plt.show()

#### Consolidate metrics for a ranked composite semantic dynamics index using PCA

In [ ]:
# assemble our five feature metrics into single composite correlation matrix

score_features = []
metrics = [
    "Path Instability (Euclidean)", 
    "Cumulative Path (Euclidean)", 
    "Net Displacement (Cosine)", 
    "Neighborhood Turnover (1-Jaccard)",
    "Neighborhood Dispersion (Cosine)"
]

for word in valid_words:
    row = [
        path_instability_scores[word],
        cumulative_dist_scores[word],
        displacement_scores[word],
        1.0 - np.mean(jaccard_scores[word]), # inverted here so that all follow: higher = more dynamic
        np.mean(dispersion_scores[word])
    ]
    score_features.append(row)

feature_matrix = np.array(score_features)
corr_matrix = np.corrcoef(feature_matrix, rowvar=False)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, xticklabels=wrap_text(metrics), yticklabels=wrap_text(metrics), vmin=-1, vmax=1)
plt.title("Correlation Matrix of Semantic Dynamics Metrics")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# standardize feature matrix and use SVD with PCA to extract dominant semantic variation

r = len(metrics)

# standardize feature matrix
feature_matrix_std = standardize(feature_matrix)[0]

# perform SVD to get 4 singular values for explained variance
# U, S, V_T = np.linalg.svd(feature_matrix_std, full_matrices=True)
U, S, V_T = sing_val_decomp(feature_matrix_std, r=r, mode="full")

# calculate explained variance for first principal component 
# sigma_sq = S**2
sigma_sq = [S[i][i]**2 for i in range(r)]
total_var = sum(sigma_sq)
explained_pc1_var = sigma_sq[0] / total_var

print(f"Variance explained by PC1: {explained_pc1_var * 100:.2f}%\n")

In [ ]:
# extract first principal component vector and plot its loadings

pc1 = V_T[0, :]
if np.sum(pc1) < 0: pc1 = -pc1

plt.figure(figsize=(8, 5))
plt.bar(wrap_text(metrics), pc1, color='salmon')
plt.axhline(0, color='black', linewidth=1)
plt.title(f"PC1 Loadings (5-metric Semantic Dynamics Index Vector)\nExplains {explained_pc1_var * 100:.1f}% of Variance")
plt.ylabel("Weight in PC1")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# composite index ranking

composite_scores = feature_matrix_std @ pc1
composite_index = {valid_words[i]: float(composite_scores[i]) for i in range(len(valid_words))}
composite_index_sorted = sorted(composite_index.items(), key=lambda x: x[1], reverse=True)

N = 20 # num words to display for top/bottom composite index ranking

# extract top N most dynamic words
top_dynamic_entries = composite_index_sorted[:N]
top_words, top_scores = zip(*top_dynamic_entries)

# extract bottom N least dynamic words
bottom_dynamic_entries = composite_index_sorted[-N:]
bottom_words, bottom_scores = zip(*bottom_dynamic_entries)

# find max score across either top/bottom tails for symmetric horizontal scaling
interval_step = 0.5
top_scores_arr = np.asarray(top_scores, dtype=float)
bottom_scores_arr = np.asarray(bottom_scores, dtype=float)
half_extent = float(max(np.max(top_scores_arr), np.max(np.abs(bottom_scores_arr))))
max_score_rounded = math.ceil(half_extent / interval_step) * interval_step

x_pad = max(interval_step, 0.08 * max_score_rounded)
x_min = -max_score_rounded - x_pad
x_max = max_score_rounded + x_pad

# plot final rankings
fig, ax = plt.subplots(2, 1, figsize=(10, 10))

# plotting top N most dynamic words
ax[0].barh(top_words, top_scores, color='salmon')
ax[0].invert_yaxis()
ax[0].set_title('Top 20 Highest Semantically Dynamic Words (Most Shift)')
ax[0].set_xlabel('Composite Score (PC1)')
ax[0].set_ylabel('Words')
ax[0].set_xlim(x_min, x_max)
ax[0].axvline(0, color='black', linewidth=1)

# plotting bottom N least dynamic words
ax[1].barh(bottom_words, bottom_scores, color='cadetblue')
ax[1].invert_yaxis()
ax[1].set_title('Bottom 20 Lowest Semantically Dynamic Words (Most Stable)')
ax[1].set_xlabel('Composite Score (PC1)')
ax[1].set_ylabel('Words')
ax[1].set_xlim(x_min, x_max)
ax[1].axvline(0, color='black', linewidth=1)

plt.tight_layout()
plt.show()